<a href="https://colab.research.google.com/github/Marcin19721205/Timeseries_Data_Processing_Basic/blob/main/LSTM_Basic_03_Shape.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import tensorflow as tf
print(tf.__version__)


2.19.0


In [2]:
from tensorflow.keras.layers import Input, SimpleRNN, Dense, Flatten
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import SGD, Adam

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

#Things to be memorized
## N - number of samples
## T - sequence Lenght
## D - Number of Input features
## M - Number of hidden Units
## K - Number of output units

In [8]:
# Make some data
N = 1
T = 10
D = 3
K = 2 # still regression problem
X = np.random.randn(N, T, D)

In [9]:
# Make a RNN  # budujemy najprostszy model RNN: sekwencja (T,D) -> wyjście (K)

M = 5  # number of hidden units  # liczba neuronów w stanie ukrytym RNN (rozmiar wektora pamięci)
i = Input(shape=(T, D))  # macierz T×D na próbkę: T kroków czasu, D cech na krok (batch jest domyślnie None)
x = SimpleRNN(M)(i)  # RNN czyta sekwencję i zwraca ostatni stan ukryty: shape=(M,) (domyślnie return_sequences=False) - activation = default = Tanh
x = Dense(K)(x)  # output  # warstwa wyjściowa: mapuje stan (M,) -> K wartości (np. K=1 regresja, K=liczba klas)
model = Model(i, x)  # składamy model: Input -> SimpleRNN -> Dense


In [10]:
# Get the output
Yhat = model.predict(X) # to check of output shape is as expected
print(Yhat)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 259ms/step
[[-0.23403662 -0.5135005 ]]


In [11]:
# See if we can replicate this output
# Get the weight first
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 10, 3)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_1 (SimpleRNN)        │ (None, 5)              │            45 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 2)              │            12 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 57 (228.00 B)

 Trainable params: 57 (228.00 B)

 Non-trainable params: 0 (0.00 B)

In [15]:
model.layers[1].get_weights()
# matrix 3x5 (DxM)
# matrix 5x5
# 5 lenght vector

[array([[-0.3835475 , -0.20947272,  0.81237394,  0.3650065 ,  0.32820576],
        [-0.2838933 ,  0.45482224,  0.43266338, -0.48386663,  0.74994236],
        [ 0.25008172,  0.38579255,  0.41723996,  0.7362252 ,  0.03395075]],
       dtype=float32),
 array([[ 0.3824737 ,  0.2259976 , -0.19178714,  0.3437546 ,  0.80479157],
        [ 0.60840344,  0.45827627, -0.28726625,  0.16511407, -0.5568155 ],
        [ 0.23334637, -0.6717949 , -0.677656  , -0.18710685, -0.00381681],
        [-0.5083756 , -0.02519216, -0.36022732,  0.76443666, -0.16368443],
        [-0.41311353,  0.535691  , -0.54010606, -0.4849721 ,  0.12433799]],
       dtype=float32),
 array([0., 0., 0., 0., 0.], dtype=float32)]

In [16]:
#check their shapes
#should make sense
#First output is input > hidden
#second output is hidden layer
#third output is bias term (vector of lenght M)
a,b,c = model.layers[1].get_weights()
print(a.shape, b.shape, c.shape)

(3, 5) (5, 5) (5,)


In [18]:
Wx, Wh, bh = model.layers[1].get_weights()
Wo, bo = model.layers[2].get_weights()

In [20]:
h_last = np.zeros(M) #initial hidden state
x = X[0] #the one and only sample
Yhats = [] #where we store the outputs

for t in range(T):
  h=np.tanh(x[t].dot(Wx)+h_last.dot(Wh)+bh)
  y=h.dot(Wo)+bo #only 1 output neuron
  Yhats.append(y)

  #important: assign h to h_last
  h_last = h

#print the final output
print(Yhats[-1])

[-0.23403653 -0.51350037]
